# 13 — Hybrid Search (BM25 + Dense RRF) & Conversational Memory

**By the end of this notebook, you can combine exact keyword search (BM25) with semantic vectors for rare error codes and build multi-turn RAG memory.**

### Learning Objectives
- Understand why pure vector embeddings struggle on opaque codes (`ERR_QUANTUM_TIMEOUT 9872`) compared to BM25.
- Combine Dense + BM25 rankings cleanly using Reciprocal Rank Fusion (RRF).
- Build conversational memory that rewrites follow-up pronouns (`it`, `that plan`) into standalone queries before retrieval.

### Prerequisites
- Notebooks 00–12

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Exact Keyword Precision + Semantic Understanding + Chat Memory

Semantic vectors excel on synonyms (*"monthly fee"* <-> *"$4,200/month"*), yet can under-rank exact rare identifiers (*"`ERR_QUANTUM_TIMEOUT`"* or *"port 9872"*). Conversely, keyword search (**BM25**) nails exact tokens yet misses synonyms!

**Hybrid Search** runs **BOTH** engines in parallel and fuses them via **RRF**!
Meanwhile, in real chatbots, users ask follow-ups like Turn 2: *"Wait—what does it cost per month?"* Searching *"it"* blindly fails, so a **History-Aware Rewriter** resolves `"it"` -> `"Enterprise Plan"` before searching!

```
Turn History + Follow-up --> [Query Rewriter] --> Standalone Query
                                                        |
                                   +--------------------+--------------------+
                                   v                                         v
                           [BM25 Keyword]                           [Dense Vector Store]
                                   \                                         /
                                    +------> [Reciprocal Rank Fusion] <-----+
                                                        |
                                                        v
                                              Top Hybrid Chunks!
```

## 3. Step-by-Step Build

### Step 1: Build Both a Dense Vector Store and a BM25 Keyword Index
Let's index the exact same chunks into `InMemoryVectorStore` (semantic vectors) AND `BM25Okapi` (keyword frequencies).

In [2]:
import re
from rank_bm25 import BM25Okapi
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

tok = lambda s: re.findall(r"[a-zA-Z0-9_]+", s.lower())
chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
dense_store = InMemoryVectorStore.from_documents(chunks, embeddings)
bm25 = BM25Okapi([tok(c.page_content) for c in chunks])
print(f"Built Dual Index over {len(chunks)} chunks (Dense + BM25).")
        

Built Dual Index over 33 chunks (Dense + BM25).


> **What just happened?**
> Having identical chunk lists behind both indexes keeps RRF fusion simple and exact.

### Step 2: Compare Dense-Only vs. BM25 vs. Hybrid RRF on Rare Error Codes
Watch how BM25 locks directly onto exact error token `ERR_QUANTUM_TIMEOUT` / `9872` while Hybrid RRF preserves semantic ranking too!

In [3]:
def hybrid_search(query: str, k: int = 3):
    dense_hits = dense_store.similarity_search(query, k=k * 2)
    bm25_scores = bm25.get_scores(tok(query))
    bm25_hits = [chunks[i] for i in sorted(range(len(chunks)), key=lambda i: bm25_scores[i], reverse=True)[: k * 2]]
    rrf = {}
    for lst in (dense_hits, bm25_hits):
        for rank, d in enumerate(lst, start=1): rrf[d.page_content] = rrf.get(d.page_content, 0.0) + 1.0 / (60 + rank)
    cmap = {d.page_content: d for d in dense_hits + bm25_hits}
    return [cmap[txt] for txt, _ in sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:k]]

res_err = hybrid_search("ERR_QUANTUM_TIMEOUT 9872", k=2)
print("Hybrid Rank #1 matches command:", "novasphere-edge" in res_err[0].page_content or "9872" in res_err[0].page_content)
        

Hybrid Rank #1 matches command: True


> **What just happened?**
> Hybrid search gives you exact technical string lookup without losing paraphrased semantic matching.

### Step 3: Implement Multi-Turn Conversational Memory Rewriting
Let's simulate a two-turn chat where Turn 2 uses ambiguous pronoun `'it'`, rewrite it cleanly against `chat_history`, and search!

In [4]:
chat_history = [
    ("user", "Tell me about the Enterprise Plan"),
    ("assistant", "The Enterprise Plan costs $4,200/month and supports 200 nodes."),
]
follow_up = "Does it include ZeroGravity failover and what is its timeout threshold?"
hist_str = "\n".join(f"{role}: {msg}" for role, msg in chat_history)
standalone_q = llm.invoke(f"Rewrite into a standalone question using CHAT_HISTORY:\nCHAT_HISTORY:\n{hist_str}\nFOLLOW_UP: {follow_up}").content
print("Rewritten standalone query:", standalone_q)
        

Rewritten standalone query: Does ZeroGravity include ZeroGravity failover and what is ZeroGravity's timeout threshold?


> **What just happened?**
> Resolving `'it'` -> `'Enterprise Plan'` turns vague conversational chat into crystal-clear retrieval!

### Step 4: Generate Grounded Answer for Follow-Up Turn
Now let's pass our rewritten standalone query straight into `hybrid_search` and `llm`.

In [5]:
hits = hybrid_search(standalone_q, k=3)
ctx = "\n---\n".join(d.page_content for d in hits)
ans = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {standalone_q}").content
print("Multi-turn Hybrid Answer:", ans)
        

Multi-turn Hybrid Answer: Based on the provided context: ### How does ZeroGravity Failover work, and which plan includes it? ZeroGravity Automatic Failover is exclusively included in the **Enterprise Plan** ($4,200/month). If a primary edge gateway or cluster coordinator becomes unresponsive for more than **40 milliseconds**, ZeroGravity automatically transfers fleet state ownership to a standby peer node without dropping active safety heartbeats. Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support.


> **What just happened?**
> It answers cleanly using BOTH `company_handbook.txt` (Enterprise Plan) AND `product_faq.md` (40 milliseconds)!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Compare dense vs hybrid on rare ports** — *Hint:* Search `'9872 UDP'` via `dense_store` vs `hybrid_search`.
2. **Try a follow-up on Zurich pets** — *Hint:* Test rewriting `'Are vaccinated dogs allowed there on Fridays?'` with history mentioning `'Zurich hub'`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
h2 = "user: On which days are pets allowed in the Zurich hub?\nassistant: Tuesdays and Thursdays."
q2 = llm.invoke(f"Rewrite into a standalone question using CHAT_HISTORY:\nCHAT_HISTORY:\n{h2}\nFOLLOW_UP: What about Friday for it?").content
print("Rewritten Turn 2:", q2)
        

Rewritten Turn 2: What about Friday for Zurich hub?


## 5. Common Mistakes
- Passing raw `'What about it?'` into embeddings without rewriting — vector search matches the word `'about'` instead of the topic!
- Blending raw BM25 unbounded scores directly with `[0,1]` cosine scores without normalization/RRF.
- Letting chat history grow unboundedly across 50 turns — summarize or keep only the last 3–5 turns.

## 6. Recap
- BM25 + Dense RRF solves the classic weakness of vectors on exact error codes, SKUs, and ports.
- RRF combines disparate rankers reliably without needing weight hyperparameter guesses.
- History-aware query rewriting enables natural follow-up conversations in RAG apps.

## 7. Next Notebook
Next up (`14_production_safety_and_security.ipynb`): Harden your pipeline for production with embedding caching, latency budgets, indirect prompt injection defense, and Role-Based Access Control (RBAC)!

## 8. Video Chapter Outline
1. **Where Vector Search Falls Short** — Exact tokens, port numbers, and rare error codes.
2. **BM25 + Dense Hybrid Fusion** — Dual indexing + RRF math in 10 lines.
3. **Why Chat Follow-Ups Fail Naive RAG** — The ambiguous pronoun problem.
4. **History-Aware Query Rewriter** — Rewriting multi-turn turns cleanly before search.
5. **Recap** — Ready for Production Security & Speed!